# MUSIC Imaging with a 20 x 20 UPA

Two point targets at the same range, half a metre either side of the boresight.

**Pipeline:** `witwin.core.Scene` -> `Radar.simulate` -> `Frame.range_profile`
-> `witwin.radar.processing.music_image`

The `Scene` carries no structures: this notebook is about angular resolution and
a wall would add multipath rows at other ranges. The script version, with the
same checks as assertions, is `examples/music_imaging.py`.

**MUSIC needs a noise subspace, so a thermal-noise front end is configured.** A
noiseless simulation of two coherent targets gives a covariance of rank two with
every other eigenvalue exactly zero - there is no noise subspace to find and the
eigendecomposition itself is ill conditioned.

In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(pathlib.Path.cwd().parents[0]))

from examples.music_imaging import (
    ARRAY_SIZE,
    CONFIG,
    FIELD_OF_VIEW_RAD,
    NUM_PIXELS,
    NUM_SIGNALS,
    SPATIAL_SMOOTH,
    TARGET_OFFSET_M,
    TARGET_POSITIONS_M,
    TARGET_RANGE_M,
    TARGET_RCS_M2,
    build_radar,
    build_scene,
)
from witwin.radar import PointTargets
from witwin.radar.processing import music_image

assert torch.cuda.is_available(), "the propagation solve is native CUDA only"

## 1. Front end, world and targets

In [ ]:
radar = build_radar()
targets = PointTargets(
    positions=torch.tensor(TARGET_POSITIONS_M, dtype=torch.float32, device=radar.device), rcs=TARGET_RCS_M2
)
print("array", ARRAY_SIZE, "x", ARRAY_SIZE, "=", ARRAY_SIZE * ARRAY_SIZE, "virtual elements")
print("targets", TARGET_POSITIONS_M)

## 2. Simulate one frame

In [ ]:
result = radar.simulate(build_scene(), targets, times=(0.0,), los=True, reflections=0)
print("cube", tuple(result.cube.shape), result.axis_names)
print("sampling", result.motion_sampling, "rows", result.last_radar_paths.path_count)

## 3. Range gate

`music_image` refuses to pick the range bins itself: reading a peak off a
spectrum is a modelling choice, and making it silently is how an imager ends up
imaging a sidelobe. Both targets share one range, so one gate holds both.

`result.frame(0)` pairs the frame's cube with `result.axes`, the metadata every
processing stage reads, built once by the simulation from the waveform spec and
the array the cube came from. The cube is already a range spectrum, so the range
profile is rectangular (the range stage refuses a taper on an FMCW spectrum).

In [ ]:
frame = result.frame(0)
axes = frame.axes
geometry = frame.array()
profile = frame.range_profile()

range_energy = profile.data.abs().sum(dim=(0, 1, 2))
peak_bin = int(torch.argmax(range_energy))
range_bins = torch.arange(peak_bin - 2, peak_bin + 3, device=radar.device)
print("gate bins", range_bins.tolist(), "centre", round(float(axes.range_m[peak_bin]), 4), "m")

## 4. MUSIC pseudo-spectrum

In [ ]:
grid = torch.linspace(-FIELD_OF_VIEW_RAD / 2, FIELD_OF_VIEW_RAD / 2, NUM_PIXELS, device=radar.device)
image = music_image(
    profile,
    geometry,
    elevation_rad=grid,
    azimuth_rad=grid,
    range_bins=range_bins,
    num_signals=NUM_SIGNALS,
    spatial_smooth=SPATIAL_SMOOTH,
    num_snapshots=CONFIG["chirp_per_frame"],
)
print("image", tuple(image.shape))

In [ ]:
degrees = (grid * 180.0 / math.pi).cpu().numpy()
plane = image.abs().amax(dim=2).cpu()
expected = [math.degrees(math.atan2(offset, TARGET_RANGE_M)) for offset in (-TARGET_OFFSET_M, TARGET_OFFSET_M)]

lobes = plane.amax(dim=1)
interior = lobes[1:-1]
peaks = torch.nonzero((interior > lobes[:-2]) & (interior > lobes[2:])).reshape(-1) + 1
strongest = sorted(float(degrees[index]) for index in peaks[torch.argsort(lobes[peaks], descending=True)[:2]])
print("lobes at", [round(value, 2) for value in strongest], "deg; targets at", [round(value, 2) for value in expected])

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4), dpi=72)
picture = left.imshow(
    20.0 * torch.log10(plane + 1e-30).numpy(),
    extent=[degrees[0], degrees[-1], degrees[0], degrees[-1]],
    origin="lower",
    aspect="auto",
    cmap="magma",
)
for angle in expected:
    left.axhline(angle, color="tab:cyan", ls="--", lw=0.8)
left.set_xlabel("Second image axis (deg)")
left.set_ylabel("First image axis (deg)")
left.set_title("MUSIC pseudo-spectrum")
fig.colorbar(picture, ax=left, label="dB")

right.plot(degrees, 20.0 * torch.log10(lobes + 1e-30).numpy())
for angle in expected:
    right.axvline(angle, color="tab:green", ls="--", label=f"{angle:.2f} deg")
right.set_xlabel("First image axis (deg)")
right.set_ylabel("dB")
right.set_title("Lobe profile")
right.legend()
plt.tight_layout()

The two lobes land within a pixel or two of `atan(+/-0.5 / 3.0) = +/-9.46 deg`.
The grid step here is `90 / 127 = 0.71 deg`.

The eight chirps of one frame are identical snapshots apart from the receiver
noise, because these targets carry no trajectory and this receiver declares no
oscillator phase noise: the default `Motion.auto()` resolves to static sampling
(one observation per frame), and the covariance is decorrelated by the spatial
smoothing rather than by slow time. A target with a `trajectory=` makes
`Motion.auto()` resolve to adaptive sampling, which carries the within-frame
Doppler.